# DPL Hierarchical Confidence — Evaluation Notebook

Loads the previously saved two-stage hierarchical classifier from `models/hierarchical_conf/`
and re-runs the full evaluation **without retraining**.

## Structure
1. Imports & load dataset
2. Load saved models
3. `HierarchicalDPLClassifier` class
4. Instantiate classifier
5. Evaluate — accuracy, F1, confidence distributions
6. Confidence threshold analysis
7. Group-level confidence heatmap
8. Inference helper

## 1. Imports & Load Dataset

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings("ignore")

from sklearn.metrics import accuracy_score, f1_score, classification_report

DATASETS_DIR = "../datasets"
train_df    = pd.read_csv(f"{DATASETS_DIR}/dpl_train.csv")
val_df      = pd.read_csv(f"{DATASETS_DIR}/dpl_val.csv")
test_df     = pd.read_csv(f"{DATASETS_DIR}/dpl_test.csv")
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

HIERARCHY = {
    "Finance & Treasury": ["DPL005","DPL006","DPL013","DPL014","DPL026","DPL027","DPL035","DPL043","DPL057","DPL058","DPL067","DPL075","DPL076"],
    "Staff & Employment": ["DPL048","DPL051","DPL052","DPL062","DPL068","DPL070","DPL073"],
    "Operational & Administrative": ["DPL001","DPL008","DPL016","DPL037","DPL041","DPL046","DPL047","DPL054","DPL055","DPL061","DPL063","DPL064","DPL065","DPL069"],
    "Professional & External Services": ["DPL003","DPL009","DPL012","DPL038","DPL049"],
    "Revenue & Income": ["DPL024","DPL030","DPL045","DPL066","DPL071","DPL072"],
    "Gains, Losses & Adjustments": ["DPL017","DPL018","DPL019","DPL020","DPL021","DPL022","DPL023","DPL028","DPL029"],
    "Asset & Accounting Adjustments": ["DPL002","DPL010","DPL032","DPL042","DPL050"],
    "Regulatory, Tax & Compliance": ["DPL015","DPL025","DPL031","DPL036","DPL053"],
    "Miscellaneous": ["DPL004","DPL007","DPL011","DPL033","DPL034","DPL039","DPL040","DPL044","DPL056","DPL059","DPL060","DPL074"],
}
TAG_TO_GROUP = {tag: grp for grp, tags in HIERARCHY.items() for tag in tags}

for df in [train_df, val_df, test_df, trainval_df]:
    df["group"] = df["dpl_tag"].map(TAG_TO_GROUP)
test_df = test_df.dropna(subset=["group"])

X_test   = test_df["description"].tolist()
true_tags = test_df["dpl_tag"].tolist()

print(f"Test rows : {len(test_df):,}")
print(f"Groups    : {len(HIERARCHY)}")

## 2. Load Saved Models

In [ ]:
SAVE_DIR = "../models/hierarchical_conf"
_REQUIRED = ["l1_group_classifier.joblib", "l1_group_label_encoder.joblib", "hierarchy.json"]
_model_exists = os.path.isdir(SAVE_DIR) and all(os.path.isfile(f"{SAVE_DIR}/{f}") for f in _REQUIRED)
if not _model_exists:
    raise FileNotFoundError(f"No saved models found at '{SAVE_DIR}'. Run the training notebook first.")

with open(f"{SAVE_DIR}/hierarchy.json") as f:
    _hier = json.load(f)

l1_clf   = joblib.load(f"{SAVE_DIR}/l1_group_classifier.joblib")
le_group = joblib.load(f"{SAVE_DIR}/l1_group_label_encoder.joblib")

l2_clfs, l2_les = {}, {}
for group in _hier:
    _safe = group.replace(" ", "_").replace("&", "and").replace(",", "")
    if os.path.isfile(f"{SAVE_DIR}/l2/{_safe}_clf.joblib"):
        l2_clfs[group] = joblib.load(f"{SAVE_DIR}/l2/{_safe}_clf.joblib")
        l2_les[group]  = joblib.load(f"{SAVE_DIR}/l2/{_safe}_le.joblib")

print(f"Level-1 loaded | Level-2 loaded: {len(l2_clfs)}/{len(_hier)} groups")

## 3. `HierarchicalDPLClassifier` Class

In [ ]:
class HierarchicalDPLClassifier:
    """Two-stage classifier with full confidence breakdown."""

    def __init__(self, l1_clf, le_group, l2_clfs, l2_les):
        self.l1_clf   = l1_clf
        self.le_group = le_group
        self.l2_clfs  = l2_clfs
        self.l2_les   = l2_les

    def predict(self, descriptions):
        return [r["tag"] for r in self.predict_detailed(descriptions)]

    def predict_detailed(self, descriptions, top_k=3, threshold=0.0):
        l1_probs      = self.l1_clf.predict_proba(descriptions)
        l1_pred_ids   = l1_probs.argmax(axis=1)
        l1_pred_names = self.le_group.inverse_transform(l1_pred_ids)
        l1_conf       = l1_probs.max(axis=1)

        results = [None] * len(descriptions)

        for group in set(l1_pred_names):
            indices = [i for i, g in enumerate(l1_pred_names) if g == group]
            batch   = [descriptions[i] for i in indices]

            if group not in self.l2_clfs:
                for i in indices:
                    results[i] = {"description": descriptions[i], "group": group,
                                  "group_confidence": float(l1_conf[i]),
                                  "tag": "UNKNOWN", "tag_confidence": 0.0,
                                  "joint_confidence": 0.0, "top_k_tags": [],
                                  "accepted": False}
                continue

            l2_probs    = self.l2_clfs[group].predict_proba(batch)
            l2_pred_ids = l2_probs.argmax(axis=1)
            l2_tags     = self.l2_les[group].inverse_transform(l2_pred_ids)
            l2_conf     = l2_probs.max(axis=1)

            for k, i in enumerate(indices):
                g_conf     = float(l1_conf[i])
                t_conf     = float(l2_conf[k])
                joint      = g_conf * t_conf
                top_k_tags = [
                    {"tag": self.l2_les[group].classes_[j],
                     "tag_confidence": float(l2_probs[k, j]),
                     "joint_confidence": g_conf * float(l2_probs[k, j])}
                    for j in np.argsort(l2_probs[k])[::-1][:top_k]
                ]
                results[i] = {
                    "description": descriptions[i],
                    "group": group, "group_confidence": g_conf,
                    "tag": l2_tags[k], "tag_confidence": t_conf,
                    "joint_confidence": joint, "top_k_tags": top_k_tags,
                    "accepted": joint >= threshold,
                }
        return results

## 4. Instantiate Classifier

In [ ]:
hier_clf = HierarchicalDPLClassifier(l1_clf, le_group, l2_clfs, l2_les)
print("HierarchicalDPLClassifier ready.")

## 5. Evaluate

In [ ]:
t0 = time.time()
detailed    = hier_clf.predict_detailed(X_test, top_k=3)
infer_time  = time.time() - t0

hier_preds  = [r["tag"]             for r in detailed]
grp_confs   = np.array([r["group_confidence"]  for r in detailed])
tag_confs   = np.array([r["tag_confidence"]    for r in detailed])
joint_confs = np.array([r["joint_confidence"]  for r in detailed])

hier_acc = accuracy_score(true_tags, hier_preds)
hier_f1  = f1_score(true_tags, hier_preds, average="weighted", zero_division=0)

print(f"Inference time        : {infer_time:.2f}s")
print(f"Accuracy              : {hier_acc:.4f}")
print(f"Weighted F1           : {hier_f1:.4f}")
print()
print(f"Mean group confidence : {grp_confs.mean():.4f}")
print(f"Mean tag confidence   : {tag_confs.mean():.4f}")
print(f"Mean joint confidence : {joint_confs.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
pairs = [
    (grp_confs,   "Group confidence P(group)",               "steelblue"),
    (tag_confs,   "Tag confidence P(tag|group)",             "darkorange"),
    (joint_confs, "Joint confidence P(group)×P(tag|group)", "green"),
]
for ax, (conf, title, color) in zip(axes, pairs):
    ax.hist(conf, bins=40, color=color, edgecolor="white", linewidth=0.4)
    ax.axvline(0.50, color="red",    linestyle="--", linewidth=1.2, label="0.50")
    ax.axvline(0.80, color="orange", linestyle="--", linewidth=1.2, label="0.80")
    ax.set_title(title); ax.set_xlabel("Confidence"); ax.legend(fontsize=8)
axes[0].set_ylabel("Count")
plt.suptitle("Confidence Score Distributions")
plt.tight_layout(); plt.show()

In [ ]:
correct_mask = np.array(hier_preds) == np.array(true_tags)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (conf, title) in zip(axes, [
    (grp_confs,   "Group confidence"),
    (tag_confs,   "Tag confidence"),
    (joint_confs, "Joint confidence"),
]):
    ax.hist(conf[correct_mask],  bins=30, alpha=0.6, label="Correct",   color="green")
    ax.hist(conf[~correct_mask], bins=30, alpha=0.6, label="Incorrect", color="red")
    ax.set_title(title); ax.set_xlabel("Confidence"); ax.legend()
axes[0].set_ylabel("Count")
plt.suptitle("Confidence on Correct vs Incorrect Predictions")
plt.tight_layout(); plt.show()

## 6. Confidence Threshold Analysis

In [ ]:
true_arr = np.array(true_tags)
pred_arr = np.array(hier_preds)

thresholds = np.arange(0.05, 1.0, 0.05)
rows = []
for t in thresholds:
    mask = joint_confs >= t
    if mask.sum() == 0: break
    rows.append({"threshold": t, "coverage": mask.mean(),
                 "accuracy": accuracy_score(true_arr[mask], pred_arr[mask]),
                 "abstentions": (~mask).sum()})
thresh_df = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(thresh_df["threshold"], thresh_df["accuracy"], "o-", color="green", linewidth=2)
ax1.axvline(0.50, color="red", linestyle="--", linewidth=1, label="0.50")
ax1.axvline(0.80, color="orange", linestyle="--", linewidth=1, label="0.80")
ax1.set_xlabel("Joint confidence threshold"); ax1.set_ylabel("Accuracy on accepted predictions")
ax1.set_title("Accuracy vs Threshold"); ax1.set_ylim(0.8, 1.01); ax1.legend()
ax2.plot(thresh_df["threshold"], thresh_df["coverage"] * 100, "o-", color="steelblue", linewidth=2)
ax2.axvline(0.50, color="red", linestyle="--", linewidth=1)
ax2.axvline(0.80, color="orange", linestyle="--", linewidth=1)
ax2.set_xlabel("Joint confidence threshold"); ax2.set_ylabel("% of predictions accepted")
ax2.set_title("Coverage vs Threshold"); ax2.set_ylim(0, 105)
plt.tight_layout(); plt.show()

print(f"\n{'Threshold':>10}  {'Coverage':>9}  {'Accuracy':>9}  {'Abstentions':>12}")
print("-" * 48)
for _, row in thresh_df[thresh_df["threshold"].isin([0.50, 0.60, 0.70, 0.80, 0.90])].iterrows():
    print(f"{row['threshold']:>10.2f}  {row['coverage']:>9.1%}  {row['accuracy']:>9.4f}  {int(row['abstentions']):>12}")

## 7. Group-Level Confidence Heatmap

In [ ]:
test_df_copy = test_df.copy()
test_df_copy["predicted_tag"]    = hier_preds
test_df_copy["joint_confidence"] = joint_confs
test_df_copy["correct"]          = test_df_copy["dpl_tag"] == test_df_copy["predicted_tag"]

grp_summary = test_df_copy.groupby("group").agg(
    mean_joint_conf=("joint_confidence", "mean"),
    accuracy=("correct", "mean"),
    count=("dpl_tag", "count"),
).round(4).sort_values("accuracy", ascending=False)

print(f"{'Group':<42} {'Count':>6}  {'Accuracy':>9}  {'Mean Joint Conf':>16}")
print("-" * 78)
for grp, row in grp_summary.iterrows():
    print(f"  {grp:<40} {int(row['count']):>6}  {row['accuracy']:>9.4f}  {row['mean_joint_conf']:>16.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(grp_summary["mean_joint_conf"], grp_summary["accuracy"],
    s=grp_summary["count"] * 0.5, c=grp_summary["accuracy"], cmap="RdYlGn",
    vmin=0.7, vmax=1.0, edgecolors="black", linewidths=0.5, alpha=0.8)
for grp, row in grp_summary.iterrows():
    ax.annotate(grp[:20], (row["mean_joint_conf"], row["accuracy"]), fontsize=7, ha="center", va="bottom")
ax.set_xlabel("Mean joint confidence"); ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs Confidence by Group (bubble size ∝ sample count)")
plt.colorbar(scatter, ax=ax, label="Accuracy"); plt.tight_layout(); plt.show()

## 8. Inference Helper

In [ ]:
test_descs = [
    "INV-55234 – Deloitte audit services FY2025",
    "Monthly payroll – March 2026 – Finance",
    "Interest charged on HSBC overdraft – January",
    "Interest received on shareholder loan – April 2025",
    "Google Ads campaign – Q2 2026",
    "Office rent – London HQ – April 2026",
    "Redundancy costs – IT department – 2025",
    "AWS cloud hosting – February subscription",
    "FX loss on USD settlement – AP-78341",
    "Miscellaneous expense",
]

inf_results = hier_clf.predict_detailed(test_descs, top_k=3, threshold=0.5)
print(f"{'Description':<50} {'Group':<32} G.Conf  {'Tag':<8} T.Conf  Joint   Status")
print("-" * 120)
for r in inf_results:
    status = "✓ AUTO" if r["accepted"] else "⚠ REVIEW"
    print(f"{r['description'][:49]:<50} {r['group'][:31]:<32} "
          f"{r['group_confidence']:>6.3f}  {r['tag']:<8} "
          f"{r['tag_confidence']:>6.3f}  {r['joint_confidence']:>6.3f}  {status}")

In [ ]:
desc   = "Miscellaneous expense"
detail = hier_clf.predict_detailed([desc], top_k=5, threshold=0.5)[0]
print(f"Description  : {detail['description']}")
print(f"Group        : {detail['group']}  (confidence: {detail['group_confidence']:.4f})")
print(f"Status       : {'✓ AUTO' if detail['accepted'] else '⚠ REVIEW'}")
print()
print(f"{'Rank':<5} {'Tag':<10} {'P(tag|group)':>13} {'Joint P':>9}")
print("-" * 42)
for rank, t in enumerate(detail["top_k_tags"], 1):
    print(f"{rank:<5} {t['tag']:<10} {t['tag_confidence']:>13.4f} {t['joint_confidence']:>9.4f}")